# R11 — Hybrid RAG over the SOP Binder + Gold Answers + Evaluation

`answer_sop_question(q)` retrieves the top SOP chunks from the Delta Sync index (hybrid = semantic + BM25, so "Instant" questions pull the Instant SOP and "Standard" questions the Standard SOP), reranks them with the chat model, and answers only from those chunks — replying exactly `not in the SOPs` when the binder has no answer. Builds `hop.genai.gold_answers` (G01-G12 of gold_set.json plus UQ1/UQ2) and `evidence/r11_eval.json` (hit-rate@5, MRR, LLM judge scores).

In [0]:
# Config cell — change model names here only, if a built-in model name changes
CATALOG = "hop"
CHAT_MODEL = "databricks-meta-llama-3-3-70b-instruct"
EMBED_MODEL = "databricks-gte-large-en"
AS_OF_DATE = "2026-08-31"
VS_ENDPOINT = "hungryhop-vs"
VS_INDEX = f"{CATALOG}.genai.sop_chunks_idx"
EVIDENCE_PATH = "/Workspace/Users/divi44246@gmail.com/Hungry-Hop/evidence/r11_eval.json"
print(CATALOG, CHAT_MODEL, EMBED_MODEL, AS_OF_DATE, VS_INDEX)

## Design

1. **Retrieval** — hybrid `query_index` (ANN + BM25 keyword) over `hop.genai.sop_chunks_idx`, top 10. BM25 keeps class-specific terms (Instant/Standard) in play, which is what the 03-vs-04 citation rule needs.
2. **LLM rerank** — the chat model orders the 10 retrieved chunks best-first for the question (temperature 0, structured output). Top 5 survive. If the rerank call fails, the hybrid order is kept as-is.
3. **Grounded generation** — the chat model answers using only the 5 chunks; citations are chunk ids; when the binder does not answer, the answer is exactly `not in the SOPs`.
4. **Gold answers** — `hop.genai.gold_answers`: question / answer / cited_chunk_ids (array, bare chunk ids, best first, up to 5) for G01-G12 of `gold_set.json` plus UQ1 ("Does Hungryhop deliver alcohol?") and UQ2 ("What is the monthly salary of a Hungryhop rider?"), which must answer `not in the SOPs`.
5. **Evaluation** — retrieval quality over G01-G18 against `expected_chunks` (G19 is the multi-hop sentinel and is excluded per its own note): hit-rate@5 and MRR; gates hit-rate@5 >= 0.80, MRR >= 0.70. An LLM judge scores every gold answer against the expected facts. Results land in `evidence/r11_eval.json` with a plain error field if the judges did not run.

In [0]:
# R11a — answer_sop_question: hybrid retrieval -> LLM rerank -> grounded answer
from databricks.sdk import WorkspaceClient
import json, re

w = WorkspaceClient()

def _llm(prompt):
    """One chat-model call via ai_query, temperature 0. Returns (text, error)."""
    spark.createDataFrame([(prompt,)], ["prompt"]).createOrReplaceTempView("r11_prompt")
    sql = (f"SELECT ai_query('{CHAT_MODEL}', prompt, "
           f"modelParameters => named_struct('temperature', 0.0), failOnError => false) AS res "
           f"FROM r11_prompt")
    res = spark.sql(sql).collect()[0][0]
    if getattr(res, "errorMessage", None):
        return None, str(res.errorMessage)
    raw = getattr(res, "result", None)
    if raw is None:
        return None, "empty model result"
    return raw, None

_RERANK_SCHEMA = "STRUCT<order:STRING>"          # comma-separated chunk ids, best first
_ANSWER_SCHEMA = "STRUCT<answer:STRING, cited_chunk_ids:STRING, not_in_sops:BOOLEAN>"

def _split_ids(s):
    return [x.strip() for x in (s or "").split(",") if x.strip()]

def _extract_json(text):
    """Pull the first {...} JSON object out of a model reply."""
    m = re.search(r"\{.*\}", text, re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except Exception:
        return None

def _retrieve(q, k=10):
    res = w.vector_search_indexes.query_index(
        index_name=VS_INDEX, columns=["chunk_id", "heading", "text"],
        query_text=q, query_type="HYBRID", num_results=k)
    out = []
    for r in res.result.data_array:
        out.append({"chunk_id": r[0], "heading": r[1], "text": r[2]})
    return out

def _rerank(q, chunks):
    listing = "\n".join(
        f"- {c['chunk_id']} | {c['heading']} | {c['text'][:300].replace(chr(10), ' ')}..."
        for c in chunks)
    prompt = ("You rerank SOP sections for a support-copilot retrieval step.\n"
              f"Question: {q}\n\nCandidate sections:\n{listing}\n\n"
              "Reply with ONLY the chunk ids ordered best-first by relevance to the "
              "question, as a single comma-separated line. Keep every chunk exactly once.")
    text, err = _llm(prompt)
    if err or not text:
        return chunks  # graceful fallback: keep the hybrid order
    by_id = {c["chunk_id"]: c for c in chunks}
    wanted = _split_ids(text.replace("\n", " "))
    ordered = [by_id[cid] for cid in wanted if cid in by_id]
    ordered += [c for c in chunks if c not in ordered]
    return ordered

def answer_sop_question(q):
    """Answer a question from the SOP binder with citations.

    Returns: {question, answer, cited_chunk_ids (bare ids, best first, up to 5),
              top5 (the final ranking), rank (1-based position of the first citation),
              not_in_sops}. When the SOPs do not answer, answer is exactly
              'not in the SOPs' and cited_chunk_ids is empty.
    """
    chunks = _rerank(q, _retrieve(q, 10))[:5]
    context = "\n\n".join(f"[{c['chunk_id']}]\n{c['text']}" for c in chunks)
    prompt = (
        "You are the HungryHop support copilot. Answer the question using ONLY the "
        "SOP sections below. Quote exact numbers and rules from the sections.\n"
        "Rules:\n"
        "- In the JSON field cited_chunk_ids list the chunk id(s) you used, best first, "
        "at most 5.\n"
        "- If the sections do not answer the question, set not_in_sops true, make "
        "answer exactly the phrase: not in the SOPs, and cite nothing.\n"
        "- Reply with ONLY a JSON object, no prose around it:\n"
        '{"answer": "...", "cited_chunk_ids": ["..."], "not_in_sops": false}\n\n'
        f"Question: {q}\n\nSOP sections:\n{context}")
    text, err = _llm(prompt)
    parsed = _extract_json(text) if text else None
    if err or parsed is None or "answer" not in parsed:
        return {"question": q, "answer": "not in the SOPs", "cited_chunk_ids": [],
                "top5": [c["chunk_id"] for c in chunks], "rank": None,
                "not_in_sops": True, "error": err or "unparseable model reply"}
    top5 = [c["chunk_id"] for c in chunks]
    raw_cited = parsed.get("cited_chunk_ids") or []
    if isinstance(raw_cited, str):
        raw_cited = _split_ids(raw_cited)
    cited = [cid for cid in raw_cited if cid in top5][:5]
    not_in_sops = bool(parsed.get("not_in_sops")) or parsed["answer"].strip().lower() == "not in the sops"
    if not_in_sops:
        cited = []
    return {"question": q,
            "answer": "not in the SOPs" if not_in_sops else parsed["answer"].strip(),
            "cited_chunk_ids": cited, "top5": top5,
            "rank": (top5.index(cited[0]) + 1) if cited else None,
            "not_in_sops": not_in_sops}

# smoke test — an Instant question must surface the Instant SOP first
demo = answer_sop_question("What base coupon does a late Instant order earn?")
print(json.dumps(demo, indent=1)[:900])

In [0]:
# R11b — hop.genai.gold_answers: G01-G12 of gold_set.json + UQ1 + UQ2
import json

gold = json.load(open("/Volumes/hop/raw/landing/gold_set.json"))
gold_qs = {q["id"]: q for q in gold["questions"]}

uq = [
    {"id": "UQ1", "question": "Does Hungryhop deliver alcohol?"},
    {"id": "UQ2", "question": "What is the monthly salary of a Hungryhop rider?"},
]

rows = []
for qid in [f"G{i:02d}" for i in range(1, 13)] + [u["id"] for u in uq]:
    qtext = gold_qs[qid]["question"] if qid in gold_qs else next(u["question"] for u in uq if u["id"] == qid)
    res = answer_sop_question(qtext)
    rows.append({"id": qid, "question": qtext, "answer": res["answer"],
                 "cited_chunk_ids": res["cited_chunk_ids"],
                 "not_in_sops": res["not_in_sops"]})
    print(qid, "|", res["answer"][:80], "|", res["cited_chunk_ids"][:2])

schema = "id STRING, question STRING, answer STRING, cited_chunk_ids ARRAY<STRING>, not_in_sops BOOLEAN"
ga = spark.createDataFrame(rows, schema).select("id", "question", "answer", "cited_chunk_ids", "not_in_sops")
ga.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.genai.gold_answers")
print("gold_answers rows:", spark.table(f"{CATALOG}.genai.gold_answers").count())
display(spark.table(f"{CATALOG}.genai.gold_answers").select("id", "answer", "cited_chunk_ids"))

In [0]:
# R11c — evaluation: hit-rate@5 + MRR over G01-G18, LLM judge, evidence/r11_eval.json
from datetime import datetime, timezone

JUDGE_SCHEMA = "STRUCT<score:DOUBLE, verdict:STRING>"
# --- 1. retrieval quality over the 18 anchor questions (G19 sentinel excluded) ---
anchor_ids = [f"G{i:02d}" for i in range(1, 19)]
hits, rr_scores, per_q = [], [], {}
for qid in anchor_ids:
    q = gold_qs[qid]
    res = answer_sop_question(q["question"])
    expected = set(q["expected_chunks"])
    top5 = res["top5"]
    hit = bool(expected & set(top5))
    positions = [top5.index(cid) + 1 for cid in expected if cid in top5]
    rr = (1.0 / min(positions)) if positions else 0.0
    hits.append(1.0 if hit else 0.0)
    rr_scores.append(rr)
    per_q[qid] = {"hit@5": hit, "rr": rr, "cited": res["cited_chunk_ids"],
                  "top5": top5, "answer": res["answer"][:200]}
    print(f"{qid} hit@5={hit} rr={rr:.2f} top1={top5[0] if top5 else None}")

hit_rate_at_5 = sum(hits) / len(hits)
mrr = sum(rr_scores) / len(rr_scores)
print(f"\nhit_rate_at_5={hit_rate_at_5:.3f}  mrr={mrr:.3f}  (n={len(hits)})")

# --- 2. LLM judge over the 14 gold answers ---
judge_error = None
judge_rows = []
try:
    for r in rows:
        facts = gold_qs[r["id"]]["expected_facts"] if r["id"] in gold_qs else (
            ["The answer must say the SOPs do not cover this"] if r["not_in_sops"] else [])
        jp = ("You judge a support-copilot answer.\n"
              f"Question: {r['question']}\n"
              f"Expected facts (ground truth): {'; '.join(facts) if facts else 'none recorded'}\n"
              f"Answer to judge: {r['answer']}\n\n"
              "Score 1.0 if the answer states the expected facts (or, for a fact not covered "
              "by the SOPs, correctly says 'not in the SOPs'); 0.0 if it is wrong, invents "
              "rules, or answers an out-of-scope question from imagination.\n"
              "Reply with ONLY a JSON object, no prose around it:\n"
              '{"score": 1.0, "verdict": "one short sentence"}\n')
        text, err = _llm(jp)
        parsed = _extract_json(text) if text else None
        if err or parsed is None or "score" not in parsed:
            raise RuntimeError(f"judge failed on {r['id']}: {err or text}")
        judge_rows.append({"id": r["id"], "score": float(parsed["score"]),
                           "verdict": str(parsed.get("verdict", ""))})
        print(f"judge {r['id']}: {parsed['score']} | {str(parsed.get('verdict', ''))[:70]}")
except Exception as e:
    judge_error = str(e)
    print("JUDGE ERROR:", judge_error)

# --- 3. evidence/r11_eval.json ---
status = "pass" if (hit_rate_at_5 >= 0.80 and mrr >= 0.70) else "fail"
evidence = {
    "hit_rate_at_5": round(hit_rate_at_5, 4),
    "mrr": round(mrr, 4),
    "status": status,
    "ran": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "judges": judge_rows,
}
if judge_error:
    evidence["error"] = judge_error
with open(EVIDENCE_PATH, "w") as f:
    json.dump(evidence, f, indent=1)
print("status:", status, "-> evidence written to", EVIDENCE_PATH)
print("per-question detail:", json.dumps(per_q, indent=1)[:2000])